## 01 - Neural Classification - Original
In this notebook, we will do a neural architecture with the original dataset.
This model contains:
- Bert embeddings (using sentence-transformer library)
- Neural (Multi-layer perceptron) classifier from Sk-learn

In [1]:
# !pip install sentence-transformers

In [2]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from sklearn.metrics import precision_recall_fscore_support, f1_score, classification_report
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import cross_val_score

In [3]:
embed_model_name = "LaBSe"

In [4]:
data_path = "../../../data/" 

# please point to correct downloaded transcribed folders of the original dataset
train_fname = data_path + "splits/train_50pc.csv"
dev_fname = data_path + "splits/dev_50pc.csv"


train_df = pd.read_csv(train_fname)
dev_df = pd.read_csv(dev_fname)

print("Train len =", len(train_df))
print("Dev df =", len(dev_df))

Train len = 83
Dev df = 83


In [5]:
train_df.head()

,file_id,text,target
0,adrso021,"Everything you see going on in this picture, g...",CN
1,adrso265,"Now you just look at the picture. Okay, this i...",CN
2,adrso237,Tell me everything that you see going on in th...,AD
3,adrso197,Girls and boys getting her one that stores sto...,AD
4,adrso017,"All right, we have a few more tasks to go thro...",CN


In [6]:
dev_df.head()

,file_id,text,target
0,adrso025,Tell me everything that's going on\n\n\n\nthe ...,AD
1,adrso028,She will find her mother worship trying to dis...,AD
2,adrso032,What's going on well the children are climbing...,AD
3,adrso035,Okay here's the picture tell me I'll tell you ...,AD
4,adrso036,Tell me what's going on\n\n\n\nall the action ...,AD


### Pre-processing
We will lowercase the text for better efficiency

In [7]:
train_df["text_lower"] = train_df["text"].apply(lambda text: text.lower().strip())
dev_df["text_lower"] = dev_df["text"].apply(lambda text: text.lower().strip())

### Apply Sentence Transformer

We can use Sentence Transformer to convert text to a fixed-length embedding vector

In [8]:
vectorizer = SentenceTransformer(embed_model_name) 

C:\Users\Minh UBC\miniconda3\Lib\site-packages\torch\_utils.py:831: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  return self.fget.__get__(instance, owner)()


In [9]:
corpus = train_df["text_lower"]
print("Creating text embeddings...")
X_train = vectorizer.encode(corpus)
print("Completed")

Creating text embeddings...
Completed


In [10]:
print(X_train.shape)

(83, 768)


### Train Neural Classifier

We can now train a neural logistic regression classifier using processed X_train

In [11]:
y_train = train_df["target"]

print("Training neural classifier...")
hidden_size = 500
clf = MLPClassifier(solver="lbfgs",
                    alpha=1e-5,
                    hidden_layer_sizes=(hidden_size, hidden_size, hidden_size),
                    learning_rate_init=0.0001,
                    max_iter=500, 
                    random_state=42
                   )

clf.fit(X_train, y_train)
print("Completed")

Training neural classifier...
Completed


In [12]:
# getting score on train set
train_score = clf.score(X_train, y_train)
print(f"Train accuracy = {train_score*100:.2f} %")

Train accuracy = 100.00 %


### Get accuracy using cross validation

In [13]:
print("Performing cross-validation...")
scores = cross_val_score(clf, X_train, y_train, cv=5)
print("Completed")
print(scores)

Performing cross-validation...
Completed
[0.47058824 0.64705882 0.76470588 0.75       0.625     ]


In [14]:
print("Average =", np.average(scores))

Average = 0.6514705882352941


In [15]:
# get precision / recall / f1
y_pred = clf.predict(X_train)
print(classification_report(y_train, y_pred))

              precision    recall  f1-score   support

          AD       1.00      1.00      1.00        47
          CN       1.00      1.00      1.00        36

    accuracy                           1.00        83
   macro avg       1.00      1.00      1.00        83
weighted avg       1.00      1.00      1.00        83



### Now we can evaluate on dev set

In [16]:
X_dev = vectorizer.encode(dev_df["text_lower"])
y_dev = dev_df["target"]
print("X_dev shape =", X_dev.shape)
print("y_dev len =", len(y_dev))

X_dev shape = (83, 768)
y_dev len = 83


In [17]:
dev_score = clf.score(X_dev, y_dev)
print(f"Dev accuracy = {dev_score*100:.2f} %")

Dev accuracy = 84.34 %


In [18]:
y_pred = clf.predict(X_dev)
print(classification_report(y_dev, y_pred))

              precision    recall  f1-score   support

          AD       0.83      0.85      0.84        40
          CN       0.86      0.84      0.85        43

    accuracy                           0.84        83
   macro avg       0.84      0.84      0.84        83
weighted avg       0.84      0.84      0.84        83



In [19]:
test_fname = data_path + "splits/test_text.csv"

test_df = pd.read_csv(test_fname)
print("Test df =", len(test_df))
test_df["text_lower"] = test_df["text"].apply(lambda text: text.lower().strip())
test_df.head()

Test df = 71


,ID,Prediction,text,text_lower
0,adrsdt15,NaN,"I see oh boy, almost falling on a stool trying...","i see oh boy, almost falling on a stool trying..."
1,adrsdt40,NaN,Well the kids are trying to get the cookies an...,well the kids are trying to get the cookies an...
2,adrsdt26,NaN,"Okay, this is a family setting in a kitchen an...","okay, this is a family setting in a kitchen an..."
3,adrsdt67,NaN,"Oh, what do I see in this picture? This remind...","oh, what do i see in this picture? this remind..."
4,adrsdt58,NaN,While the young boy is reaching for a cookie j...,while the young boy is reaching for a cookie j...


In [20]:
# create X_test embeddings
print("Creating text embeddings...")
X_test = vectorizer.encode(test_df["text_lower"])
print("Completed")

Creating text embeddings...
Completed


In [21]:
test_df["Prediction"] = clf.predict(X_test)
test_df.head()

,ID,Prediction,text,text_lower
0,adrsdt15,CN,"I see oh boy, almost falling on a stool trying...","i see oh boy, almost falling on a stool trying..."
1,adrsdt40,CN,Well the kids are trying to get the cookies an...,well the kids are trying to get the cookies an...
2,adrsdt26,CN,"Okay, this is a family setting in a kitchen an...","okay, this is a family setting in a kitchen an..."
3,adrsdt67,AD,"Oh, what do I see in this picture? This remind...","oh, what do i see in this picture? this remind..."
4,adrsdt58,CN,While the young boy is reaching for a cookie j...,while the young boy is reaching for a cookie j...


In [22]:
# convert to 0/1 to be consistent to ground truth
label2binary = lambda label: "0" if label=="0" or label.lower().strip() == "cn" else "1"
assert label2binary("CN") == "0"
assert label2binary("AD") == "1"
assert label2binary("0") == "0"
assert label2binary("1") == "1"

print("Passed")

Passed


In [23]:
# apply to prediction column
test_df["Prediction"] = test_df["Prediction"].apply(label2binary)
# write to prediction output file
output_path = "../../evaluation/"
fname = "test_results_task1_neural.csv"
output_df = test_df.drop(columns=["text", "text_lower"])
output_df.head()

,ID,Prediction
0,adrsdt15,0
1,adrsdt40,0
2,adrsdt26,0
3,adrsdt67,1
4,adrsdt58,0


In [24]:
output_df.to_csv(output_path + fname,
                 index=False
                )
print("Completed")

Completed


### Output from evaluation scripts

```
$ python evaluate.py task1_gold.csv test_results_task1_neural.csv 
Loading data from task1_gold.csv...done
Loading data from test_results_task1_neural.csv...done
Gold len = 71
Prediction len = 71
Classification report:
              precision    recall  f1-score   support

           0       0.71      0.69      0.70        36
           1       0.69      0.71      0.70        35

    accuracy                           0.70        71
   macro avg       0.70      0.70      0.70        71
weighted avg       0.70      0.70      0.70        71

Accuracy = 0.704225352112676
Recall = 0.7142857142857143
```